# Notebook 13 — Thử nghiệm kiến trúc DL mới

Mỗi cell thí nghiệm tương ứng một ý tưởng theo thứ tự ưu tiên đã thống nhất. Hiện tại chỉ có cell FiLM đầu tiên; cell tổng hợp và so sánh với hai file global cũ sẽ được thêm sau khi hoàn tất các ý tưởng.

In [ ]:
from datetime import datetime
from pathlib import Path
import os
import sys
import torch

cwd = Path.cwd().resolve()
REPO_ROOT = next((p for p in (cwd, *cwd.parents) if (p / 'controllers').is_dir() and (p / 'experiments').is_dir()), None)
if REPO_ROOT is None:
    raise RuntimeError(f'Could not locate repository root from {cwd}')
sys.path.insert(0, str(REPO_ROOT))

from core.module05v2_fusion_classifier.pipeline import FusionBatchPipeline

BASE_DIR = 'D:/variant_data'
FM_PROFILE_JSON = f'{BASE_DIR}/profiling/fm_profiling_sequence_ablation.json'
FM_SUPPORT_JSON = f'{BASE_DIR}/profiling/fm_sequence_ablation_support.json'
EMBEDDING_DIR = f'{BASE_DIR}/fm_embeddings/sequence_ablation'
GEOMETRY_K = 64
GEOMETRY_DIR = f'{BASE_DIR}/geometry_v2_k{GEOMETRY_K}/{{context}}'
GEOM_PROFILE_JSON = None
GEOM_FEATURE_NAMES = [f'{m}_{f}' for m in ('dna', 'prot') for f in ('LVD_L2', 'LVD_Cosine', 'LID', 'LVD_Relative')]
TARGET_CONFIGS = [
    {'name': 'A_no_geometry', 'ablation': 'bio_core_dna_prot_spliceai', 'active_modalities': ['dna', 'prot', 'bio_core', 'spliceai'], 'geom_feature_names': None},
    {'name': 'B_with_geometry', 'ablation': 'bio_core_dna_prot_spliceai_geom', 'active_modalities': ['dna', 'prot', 'bio_core', 'spliceai', 'geom'], 'geom_feature_names': GEOM_FEATURE_NAMES},
]
DNA_MODEL = {'name': 'nt_v1_500m_dna_1001', 'seq_type': 'dna', 'sequence_length': 1001}
PROTEIN_MODEL = {'name': 'esm2_650m_prot_501', 'seq_type': 'protein', 'sequence_length': 501}
MODELS_SPACE = [DNA_MODEL, PROTEIN_MODEL]
DATASETS = [
    {'name': 'Train3Val_Test', 'train': 'train3', 'val': 'val', 'test': 'test'},
    {'name': 'Train3Val_ClinVarHQ', 'train': 'train3', 'val': 'val', 'test': 'clinvarhq'},
    {'name': 'Train3Val_UniProt', 'train': 'train3', 'val': 'val', 'test': 'uniprot'},
    {'name': 'Train3Val_ProteinGym', 'train': 'train3', 'val': 'val', 'test': 'proteingym'},
]
CONFIG = {'batch_size': 256, 'epochs': 30, 'lr': 1e-4, 'weight_decay': 1e-3, 'lr_factor': 0.5, 'lr_patience': 3, 'early_stop_patience': 6, 'num_workers': 0, 'seed': 42, 'xgb_n_jobs': 1}
EXPERIMENTS = [{'name': 'PyTorch_Concat', 'type': 'pytorch', 'fusion': 'concat'}]
EXPLAINABILITY = {'enable_shap': False, 'enable_lime': False}
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

RESUME_EXISTING_RUN = False
RESUME_BATCH_RUN_DIR = None
if RESUME_EXISTING_RUN:
    if RESUME_BATCH_RUN_DIR is None:
        raise ValueError('Set RESUME_BATCH_RUN_DIR when RESUME_EXISTING_RUN=True')
    RUN_OUTPUT_DIR = Path(RESUME_BATCH_RUN_DIR).resolve()
else:
    RUN_STAMP = datetime.now().strftime('%Y%m%d_%H%M%S')
    RUN_OUTPUT_DIR = REPO_ROOT / 'experiments' / f'module06_architecture_run_{RUN_STAMP}'
    suffix = 2
    while RUN_OUTPUT_DIR.exists():
        RUN_OUTPUT_DIR = REPO_ROOT / 'experiments' / f'module06_architecture_run_{RUN_STAMP}_{suffix:02d}'
        suffix += 1

pipeline = FusionBatchPipeline(
    base_dir=BASE_DIR, config=CONFIG, datasets=DATASETS, models_space=MODELS_SPACE,
    pooling_strategies=['center'], experiments=EXPERIMENTS, explainability=EXPLAINABILITY,
    fm_profile_json=FM_PROFILE_JSON, geom_profile_json=GEOM_PROFILE_JSON,
    geometry_dir=GEOMETRY_DIR, geom_feature_names=GEOM_FEATURE_NAMES,
    batch_run_dir=str(RUN_OUTPUT_DIR), embedding_dir=EMBEDDING_DIR,
)
print(f'Device: {DEVICE}')
print(f'Batch run directory: {pipeline.batch_run_dir}')
print(f'Target configurations: {[c["name"] for c in TARGET_CONFIGS]}')

## Ưu tiên đầu — FiLM có bio/geometry làm ngữ cảnh

Bio/geometry sinh tham số FiLM để điều chế nhánh DNA và protein. Hai nhánh được ghép với biểu diễn bio ở classifier. Chạy riêng cấu hình A và B; chọn checkpoint theo MCC validation sau tối ưu threshold. Checkpoint và CSV được lưu/resume trong batch run hiện tại.

In [ ]:
from core.module06.film_experiment import get_default_film_config, run_film_experiment

FILM_CONFIG = get_default_film_config()
# Chỉnh hyperparameter FiLM tại đây trước khi bắt đầu batch run mới.
FILM_CONTEXT = {
    'pipeline': pipeline, 'TARGET_CONFIGS': TARGET_CONFIGS, 'datasets': DATASETS,
    'DNA_MODEL': DNA_MODEL, 'PROTEIN_MODEL': PROTEIN_MODEL, 'DEVICE': DEVICE,
}
FILM_RESULTS = run_film_experiment(FILM_CONTEXT, FILM_CONFIG, display_fn=print)